<a href="https://colab.research.google.com/github/knight19720208ui/TIP_Taller_BigData/blob/main/Taller3_Retail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql import SparkSession

# Configuración del motor
spark = SparkSession.builder.appName("TIP_Retail").config("spark.sql.shuffle.partitions", "8").getOrCreate()

print("Generando 10M de compras... (Este proceso requiere algunos segundos)")
df_compras = spark.range(10000000).withColumnRenamed("id", "compra_id") \
    .withColumn("cliente_id", (F.rand(seed=30) * 300000).cast("int")) \
    .withColumn("ciudad", F.when(F.rand() < 0.4, "Bogotá").when(F.rand() < 0.7, "Medellín").otherwise("Cali")) \
    .withColumn("monto", (F.rand(seed=31) * 500 + 10).cast("double"))

# ---------------------------------------------------------
# RETO: Crear un Running Total (Suma Acumulada) y demostrar el uso de CACHE
# ---------------------------------------------------------
# PISTA 1: Defina un objeto "Window" donde particione por "cliente_id" y ordene por "compra_id" de forma ASCENDENTE.
# PISTA 2: Para lograr una suma acumulada progresiva, debe agregar un límite de filas a la ventana utilizando:
#         .rowsBetween(Window.unboundedPreceding, Window.currentRow)
# PISTA 3: Utilice F.sum("monto").over(su_ventana) para crear la columna "gasto_acumulado".
# PISTA 4 (CRÍTICO): Antes de ejecutar los filtros de validación, aplique .cache() al DataFrame resultante.
#         Inmediatamente después, fuerce su ejecución llamando a un .count(). Si omite este paso, la optimización en memoria no se activará.

# --- ESCRIBA SU CÓDIGO AQUÍ ---



# --- ZONA DE PRUEBAS (No modificar) ---
try:
    print("--- PRIMERA CONSULTA (Bogotá) ---")
    t1 = time.time()
    df_lealtad.filter(F.col("ciudad") == "Bogotá").agg(F.sum("gasto_acumulado")).show()
    print(f"Tiempo de ejecución: {time.time() - t1:.2f} segundos\n")

    print("--- SEGUNDA CONSULTA (Medellín) ---")
    t2 = time.time()
    df_lealtad.filter(F.col("ciudad") == "Medellín").agg(F.sum("gasto_acumulado")).show()
    print(f"Tiempo de ejecución: {time.time() - t2:.2f} segundos")
    print("NOTA DE EVALUACION: Si la segunda consulta tardó menos de 1 segundo, el uso de Caché fue implementado de forma correcta. Si ambas consultas tardan un tiempo similar, la instrucción .cache() o .count() fue omitida.")
except NameError:
    print("ERROR: Aún no ha creado la variable 'df_lealtad' o existe un error de sintaxis en el bloque superior.")
except Exception as e:
    print(f"ERROR en la lógica de procesamiento: {e}")